In [13]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import torch.optim as optim
import torch.nn as nn
from core.benchmarks import *
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
from core.benchmarks import *
import pandas as pd
from core.CVsplits import *
from tqdm.notebook import tqdm
import logging
from core.Log import *
import json
OUTER_FOLDS = 4; INNER_FOLDS = 3

logging.shutdown()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
setup_logger('OUTER')
# ['INNER_train', 'OUTER_train', 'OUTER_evaluate', 'Close']
logT = logging.getLogger('OUTER_train')
logE = logging.getLogger('OUTER_evaluate')
logT.info("Model;OuterFold;HP;Epoch;TrainLoss;TrainAcc;ValLoss;ValAcc;AUC;Brier;EMA_ValLoss;LR;NoImprove;LrDrop;EsTriggered;BestValLoss;BestEpoch")


In [15]:
import itertools
import json
## 1. Define all model-specific hyperparameter sweeps in one dictionary
model_configs = {
	"MultiViewCNN": {
		"LR_SWEEP": [2e-4, 5e-4],
		"DR_SWEEP": [0.2],
		"WD_SWEEP": [1e-6]
	},
#	"META+MLP": {
#		"LR_SWEEP": [2e-3, 5e-3],
#		"DR_SWEEP": [0.3],
#		"WD_SWEEP": [1e-3]
#	},
#	"RN18+MLP": {
#		"LR_SWEEP": [2e-4, 5e-4],
#		"DR_SWEEP": [0.2],
#		"WD_SWEEP": [1e-6]
#	},
#	"Axial":    {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.3]},
#	"Coronal":  {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.3]},
#	"Sagittal": {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.3]},
}

# 2. Define global parameters that are the same for all models
GLOBAL_PARAMS = {
	"P": 8,
	"Epochs": 50,
}

CV_parameters = []
ID = 1

# Iterate through each model and its specific configuration
for model_name, config in model_configs.items():

	# Generate all unique combinations of the model's hyperparameters
	# e.g., for MultiViewCNN, this will create (1e-3, 0.3, 1e-4), (1e-3, 0.4, 1e-4), etc.
	hp_combinations = list(itertools.product(
		config['LR_SWEEP'],
		config['DR_SWEEP'],
		config['WD_SWEEP']
	))

	# Loop through outer and inner folds
	for outer_fold_idx in range(1, 5):
		#for inner_fold_idx in range(1, 4):

			# Loop through each hyperparameter combination for this model
			for i, (lr, dr, wd) in enumerate(hp_combinations):
				item = {
					"Model": model_name,
					'OUTER_FOLD': outer_fold_idx,
					'HPset': i + 11,
					"LR": lr,
					"WD": wd,
					"DR": dr,
					"P": GLOBAL_PARAMS['P'],
					"Epochs": GLOBAL_PARAMS['Epochs'],
					"trained": False,
					"evaluated": False,
				}
				CV_parameters.append(item)
				ID += 1

print(f"Total combinations generated: {len(CV_parameters)}")

with open("NCV_4_3_folds/OUTER_experiments.json", "w") as f:
	json.dump(CV_parameters, f, indent=2)



Total combinations generated: 8


In [4]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, brier_score_loss
import numpy as np
import copy

logT = logging.getLogger('OUTER_train')
logE = logging.getLogger('OUTER_evaluate')


def train_OUTER_model(model, train_loader, val_loader, experiment):
	log = logging.getLogger('OUTER_train')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	epochs = experiment['Epochs']
	model_name  = experiment['Model']
	out=experiment['OUTER_FOLD']
	HP = experiment['HPset']

	LR = experiment['LR']
	WD = experiment['WD']
	P = experiment['P']
	TH = 0.5
	rel_thresh = 5e-3 if np.isclose(LR, 5e-4) else 3e-3
	sch_patience, sch_cooldown = 4, 2

	ES_PATIENCE = max(P, sch_patience + sch_cooldown + 2)  # ≥ 8
	alpha_ema = 0.30  # smoothing for EMA of val loss

	optimizer = optim.Adam(model.parameters(), lr= LR, weight_decay=WD)
	scheduler = ReduceLROnPlateau(optimizer, mode='min',
								  patience=sch_patience, factor=0.5,
								  threshold=rel_thresh, threshold_mode='rel',
								  cooldown=sch_cooldown, min_lr=1e-6)
	criterion = nn.BCEWithLogitsLoss()

	# --- best trackers ---
	best_val_loss = np.inf
	best_epoch    = -1
	best_model_state = None
	best_lr_at_best = LR
	auc_at_best   = np.nan
	brier_at_best = np.inf

	# --- EMA & patience ---
	ema_val = None
	best_ema = np.inf
	no_improve = 0

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)
	#log.info("Model;ExpID;OuterFold;HPset;Epoch;TrainLoss;TrainAcc;ValLoss;ValAcc;AUC;Brier;EMA_ValLoss;LR;NoImprove;LrDrop;EsTriggered;BestValLoss;BestEpoch;WallTimeSec\n")

	print(f" train_N: {train_N}, val_N: {val_N}		↳ Training model... ")
	for epoch in range(epochs):
		model.train()
		running_loss = 0.0
		y_true, y_pred = [], []
		for batch in train_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)

			optimizer.zero_grad(set_to_none=True)

			logits = model(axi, cor, sag, met)
			T_loss = criterion(logits, lbl)

			T_loss.backward()
			optimizer.step()

			running_loss += T_loss.item() * lbl.size(0)
			with torch.no_grad():
				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()
				y_true.append(lbl.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		TrainLoss = running_loss / max(1, train_N)
		y_true = np.concatenate(y_true).reshape(-1)
		y_pred = np.concatenate(y_pred).reshape(-1)
		TrainAcc  = (y_true == y_pred).mean()

		model.eval()
		running_loss = 0.0
		y_true, y_prob, y_pred = [], [], []

		with torch.no_grad():
			for batch in val_loader:
				axi = batch["axial_image"].to(device)
				cor = batch["coronal_image"].to(device)
				sag = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				logits = model(axi, cor, sag, met)
				V_loss = criterion(logits, lbl)
				running_loss += V_loss.item() * lbl.size(0)

				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()

				y_true.append(lbl.detach().cpu().numpy())
				y_prob.append(probs.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		ValLoss = running_loss / max(1, val_N)

		y_true  = np.concatenate(y_true).reshape(-1)
		y_prob  = np.concatenate(y_prob).reshape(-1)
		y_pred  = np.concatenate(y_pred).reshape(-1)

		ValAcc  = (y_true == y_pred).mean()
		Brier = brier_score_loss(y_true, y_prob)
		AUC = roc_auc_score(y_true, y_prob)

		# -------------- EMA + scheduler --------------
		ema_val = ValLoss if ema_val is None else alpha_ema*ValLoss + (1 - alpha_ema)*ema_val
		prev_lr = optimizer.param_groups[0]['lr']
		scheduler.step(ema_val)  # schedule on EMA, not raw ValLoss
		new_lr = optimizer.param_groups[0]['lr']
		lr_drop = int(new_lr < prev_lr)

		# -------------- early stopping test -----------
		improved = ema_val < best_ema * (1 - rel_thresh)
		if improved:
			best_ema = ema_val
			best_val_loss = ValLoss
			best_epoch = epoch
			best_lr_at_best = new_lr
			auc_at_best = AUC
			brier_at_best = Brier
			best_model_state = copy.deepcopy(model.state_dict())
			no_improve = 0
		else:
			no_improve += 1

		es_triggered = int(no_improve >= ES_PATIENCE)
		line=f"{model_name};{out};{HP};{epoch};{TrainLoss};{TrainAcc};{ValLoss};{ValAcc};{AUC};{Brier};{ema_val};{new_lr};{no_improve};{lr_drop};{es_triggered};{best_val_loss};{best_epoch}"
		log.info(line)
		if es_triggered: break

# ----- final return (best state + summary for outer fold) -----
	summary = {
        "Model": model_name,
        "OuterFold": out,
        "LR": LR,
        "WD": WD,
        "BestEpoch": best_epoch,
        "BestValLoss": float(best_val_loss),
        "AUC_at_Best": float(auc_at_best) if auc_at_best is not None else np.nan,
        "Brier_at_Best": float(brier_at_best) if brier_at_best is not None else np.nan,
        "LR_at_Best": float(best_lr_at_best),
        "ES_Patience_Used": ES_PATIENCE,
        "RelThresh": rel_thresh,
        "EMA_alpha": alpha_ema,
    }
	return summary, best_model_state



def append_experiment_results(item, path="NCV_4_3_folds/OUTER_summary.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")



In [ ]:
main_dataset = load_dataset_info(file="data/data_info.json")
DL = DataLoaderFactory(main_dataset)


In [9]:
CV_parameters = load_from_json("NCV_4_3_folds/OUTER_experiments.json")
df = pd.DataFrame(CV_parameters)
df


Loaded NCV_4_3_folds/OUTER_experiments.json.


,Model,OUTER_FOLD,HPset,LR,WD,DR,P,Epochs,trained,evaluated
0,MultiViewCNN,1,11,0.0002,0.000001,0.2,8,50,False,False
1,MultiViewCNN,1,12,0.0005,0.000001,0.2,8,50,False,False
2,MultiViewCNN,2,11,0.0002,0.000001,0.2,8,50,False,False
3,MultiViewCNN,2,12,0.0005,0.000001,0.2,8,50,False,False
4,MultiViewCNN,3,11,0.0002,0.000001,0.2,8,50,False,False
5,MultiViewCNN,3,12,0.0005,0.000001,0.2,8,50,False,False
6,MultiViewCNN,4,11,0.0002,0.000001,0.2,8,50,False,False
7,MultiViewCNN,4,12,0.0005,0.000001,0.2,8,50,False,False


In [21]:

filtered = [
	exp for exp in CV_parameters
	if exp["Model"] == "MultiViewCNN"
	and exp["OUTER_FOLD"] == 4       # 1, 2, 3, 4
	#and exp["HPset"] != 7
	#and exp["trained"] == False
]
print(f"experiments to do: {len(filtered)}")



experiments to do: 2


In [ ]:


#logE.info(f"Model;       Fold;      HPset;    CaseID;    Label;     Prediction;    Probability;      LR;  	WD; 	DR 	")


In [24]:

for experiment in filtered:
	print(experiment)
	HPset = experiment['HPset']
	OUT = 0
	model_type = experiment["Model"]
	train_loader, val_loader, _ = DL.create_outer_loaders(OUT)
	DR = experiment['DR']
	#model = SingleViewClassifier(DR)
	model = MultiViewCNN(DR)

	results, best_model_state = train_OUTER_model(model, train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models/{model_type}_F_4_HP_{HPset}.pth")
	#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	#model.to(device)
	#model.load_state_dict(best_model_state)
	#final_loss = EVALUATE_MODEL(model, test_loader, experiment)
	#results["final_loss"] = float(final_loss)

	append_experiment_results(results)





{'Model': 'MultiViewCNN', 'OUTER_FOLD': 4, 'HPset': 11, 'LR': 0.0002, 'WD': 1e-06, 'DR': 0.2, 'P': 8, 'Epochs': 50, 'trained': False, 'evaluated': False}
 train_N: 83, val_N: 10		↳ Training model... 
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 4, 'HPset': 12, 'LR': 0.0005, 'WD': 1e-06, 'DR': 0.2, 'P': 8, 'Epochs': 50, 'trained': False, 'evaluated': False}
 train_N: 83, val_N: 10		↳ Training model... 


In [13]:
append_experiment_results(results, path="NCV_5_3_folds/OUTER_results2.jsonl")


In [ ]:

def EVALUATE_MODEL(model, test_loader, experiment):
	log = logging.getLogger('OUTER_evaluate')

	TH = 0.5
	model_type = experiment["Model"]
	ExpID = experiment['ExpID']
	fold = experiment["OUTER_FOLD"]
	log = logging.getLogger('outer_eval')

	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	#model.to(device)

	model.eval()
	criterion = nn.BCEWithLogitsLoss()
	eval_N = len(test_loader.dataset)

	running_loss = 0.0
	all_predictions = []
	all_probabilities = []
	all_labels = []
	#per_case_rows = []


	#with torch.no_grad():
	with torch.inference_mode():
		print(f"	↳ Experiment {ExpID} | {model_type} | Evaluating model... ")
		for batch in test_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).float().unsqueeze(1)
			case_ids = batch["CaseID"]

			#axi_features = feature_extractor(axi)
			#cor_features = feature_extractor(cor)
			#sag_features = feature_extractor(sag)

			#combined_input = torch.cat([axi_features, cor_features, sag_features, met], dim=1)

			#logits = model(combined_input)
			#logits = model(sag, meta=met)
			#logits = model(cor, meta=met)
			#logits = model(axi, meta=met)
			logits = model(axi, sag, cor, met)
			#logits = model(meta=met)


			#loss = criterion(logits, lbl)
			loss = criterion(logits, lbl)
			running_loss += loss.item() * lbl.size(0)


			probability = torch.sigmoid(logits)
			prediction = (probability >= TH).to(torch.int32)


			prob_np = probability.squeeze(1).cpu().numpy()
			pred_np = prediction.squeeze(1).cpu().numpy()
			lbl_np  = lbl.squeeze(1).cpu().numpy()

			all_probabilities.append(prob_np)
			all_predictions.append(pred_np)
			all_labels.append(lbl_np)


			#current_batch_size = len(case_ids)


			#for i in range(current_batch_size):

			for cid, y, yhat, p in zip(case_ids, lbl_np, pred_np, prob_np):
				#per_case_rows.append({
				#	"Model": model_type,
				#	"ExpID": ExpID,
				#	"OUTER_FOLD": fold,
				#	"HPset": experiment["HPset"],
				#	"LR": experiment['LR'],
				#	"WD": experiment['WD'],
				#	"DR": experiment['DR'],
				#	"CaseID": cid,
				#	"Label": int(y),
				#	"Pred": int(yhat),
				#	"Prob": float(p),
				#	"TH": TH,
				#})

				#case_id = CaseID[i]
				#label =  lbl[i].item()
				#pred = prediction[i].item()
				#prob = probability[i].item()
				log.info(f"{model_type};  	{ExpID};  		{fold};     {experiment['HPset']};  	{cid}; 		{int(y)};		{int(yhat)};	 	{float(p):.6f}; 	 {experiment['LR']};    {experiment['WD']};  {experiment['DR']};	")

	#all_probabilities = np.concatenate(all_probabilities, axis=0)
	#all_predictions = np.concatenate(all_predictions, axis=0)
	#all_labels = np.concatenate(all_labels, axis=0)
	final_loss = running_loss / eval_N
	return final_loss




{'ExpID': 7,
 'Model': 'MultiViewCNN',
 'OUTER_FOLD': 1,
 'HPset': 200,
 'LR': 5e-05,
 'WD': 0.0002,
 'DR': 0.4,
 'best_brier': 0.2201488418747183,
 'best_LR': 5e-05,
 'brier_at_best_loss': 0.2201488418747183,
 'auc_at_best_loss': 0.8400000000000001,
 'best_V_loss': 0.6329491853713989,
 'best_th_at_best_loss': 0.4830556809902191,
 'best_epoch': 49,
 'epochs_ran': 49,
 'final_loss': 0.649279205887406}